In [1]:
# ==============================================================================
#  DYSARTHRIA DETECTION — EfficientNet-B0 & ConvNeXt-Tiny
#  Covers: Vowels (A/E/I/O/U) → Words (PAPA/PIPI/PUPU) → Sentences (S0/S1)
#  No fusion — each model evaluated independently per fold
#  Outputs:
#    • Per-fold: training curves, confusion matrix, ROC, PR curve, metrics.txt
#    • Per phoneme: OVERALL aggregated metrics (across all folds)
#    • Per task category: AVERAGE metrics table (vowels / words / sentences)
#    • Grand comparison table across all tasks and models
# ==============================================================================

# ==============================================================================
# CELL 1 — Mount Google Drive (run once)
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ==============================================================================
# CELL 2 — Install missing packages (run once per session)
# ==============================================================================
!pip install librosa --quiet

In [ ]:
# ==============================================================================
# SECTION 0 — IMPORTS
# ==============================================================================
import os
import math
import json
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')        # non-interactive backend — required for Colab
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from sklearn.metrics import (
    roc_curve, auc, classification_report, accuracy_score,
    confusion_matrix, precision_recall_curve, average_precision_score,
    f1_score
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ==============================================================================
# SECTION 1 — PATH CONFIGURATION
#
#  Drive layout expected (same merged ICMR folder as your fusion setup):
#    MyDrive/ICMR/
#        vowels/spectogram_acoustic_features.csv
#        words/spectogram_acoustic_features.csv
#        sentences/spectogram_acoustic_features.csv
#
#  Only the spectrogram CSVs are needed (pure 2-D spectrogram run, no 1-D).
#
#  <<< Only change ICMR_COLAB if your folder is NOT at MyDrive root >>>
# ==============================================================================

ICMR_COLAB   = "/content/drive/MyDrive/ICMR"
RESULTS_BASE = f"{ICMR_COLAB}/Results_EfficientNet_ConvNeXt"

VOWELS_CSV_2D    = f"{ICMR_COLAB}/vowels/spectogram_acoustic_features.csv"
WORDS_CSV_2D     = f"{ICMR_COLAB}/words/spectogram_acoustic_features.csv"
SENTENCES_CSV_2D = f"{ICMR_COLAB}/sentences/spectogram_acoustic_features.csv"

# Windows path prefixes stored inside the CSV — all map to ICMR_COLAB
# <<< Update only if your local Windows path prefix differs >>>
WINDOWS_PREFIXES = [
    r"C:\Users\frost\OneDrive\coding\Projects\speech-disorder\model_with_icmr\ICMR",
    r"C:\Users\frost\OneDrive\coding\Projects\speech-disorder\model_with_icmr_words\ICMR",
    r"C:\Users\frost\OneDrive\coding\Projects\speech-disorder\model_with_icmr_sentences\ICMR",
]

# Hyper-parameters
NUM_EPOCHS = 30
BATCH_SIZE = 16
LR         = 1e-5

# ==============================================================================
# SECTION 2 — PATH FIXER
# ==============================================================================

def fix_path(p: str) -> str:
    """Convert Windows paths stored in CSV cells to Colab-compatible paths."""
    if not isinstance(p, str):
        return p
    p = p.replace("\\", "/")
    for prefix in WINDOWS_PREFIXES:
        prefix_fwd = prefix.replace("\\", "/")
        if prefix_fwd in p:
            p = p.replace(prefix_fwd, ICMR_COLAB)
            break
    return p

# ==============================================================================
# SECTION 3 — MODELS
# ==============================================================================

def get_efficientnet_b0() -> nn.Module:
    """
    EfficientNet-B0 pretrained on ImageNet-1K.
    Compound-scaled CNN: simultaneously scales depth, width and resolution.
    Consistently outperforms MobileNetV2 / ResNet-50 at fewer parameters.
    Classifier replaced: Dropout(0.3) + Linear(1280 → 2).
    """
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_features = m.classifier[1].in_features      # 1280
    m.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, 2)
    )
    return m


def get_convnext_tiny() -> nn.Module:
    """
    ConvNeXt-Tiny pretrained on ImageNet-1K.
    Modern pure-CNN (2022) inspired by Vision Transformers:
    7×7 depthwise convolutions, LayerNorm, inverted bottleneck blocks.
    Outperforms ResNet-50 at similar FLOPs, trains very stably.
    Head replaced: LayerNorm + Linear(768 → 2).
    """
    m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    in_features = m.classifier[2].in_features      # 768
    m.classifier[2] = nn.Linear(in_features, 2)
    return m


# Registry — add more models here if needed
MODEL_REGISTRY = {
    "EfficientNet-B0": get_efficientnet_b0,
    "ConvNeXt-Tiny":   get_convnext_tiny,
}

# ==============================================================================
# SECTION 4 — DATASET
# ==============================================================================

TRANSFORM_2D = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])


class SpectrogramDataset(Dataset):
    """
    Reads pre-generated mel-spectrogram PNG files.
    Returns (3, 224, 224) float tensor, int label, and base filename.

    Required CSV columns: spectogram_file_path, Label, Phoneme, Fold, Type
    """
    def __init__(self, df: pd.DataFrame, transform=TRANSFORM_2D):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        path      = fix_path(str(row["spectogram_file_path"]))
        label     = int(row["Label"])
        base_name = os.path.splitext(os.path.basename(path))[0]

        try:
            img    = Image.open(path).convert("RGB")
            tensor = self.transform(img)
        except Exception as e:
            print(f"\n[Dataset] Load error — {path}: {e}")
            tensor = torch.zeros(3, 224, 224)

        return tensor, torch.tensor(label, dtype=torch.long), base_name

# ==============================================================================
# SECTION 5 — TRAINING ENGINE
# ==============================================================================

def train_model(model:         nn.Module,
                train_loader:  DataLoader,
                val_loader:    DataLoader,
                model_name:    str,
                save_dir:      str,
                task_name:     str,
                phoneme:       str,
                fold:          int):
    """
    Trains `model` for NUM_EPOCHS.

    Saves
    ─────
      model.pth              — state dict of the fully-trained model
      model_config.txt       — architecture + hyper-parameter record
      training_curves.png    — loss & accuracy curves

    Returns
    ───────
      val_probs  : dict  { base_name → np.ndarray shape (2,) }
      val_labels : dict  { base_name → int }
    """
    os.makedirs(save_dir, exist_ok=True)
    print(f"      [{model_name}] training …")

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)

    history = {k: [] for k in ('train_loss', 'val_loss', 'train_acc', 'val_acc')}

    for epoch in range(NUM_EPOCHS):

        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        run_loss = correct = total = 0
        for inputs, labels, _ in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            out  = model(inputs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()

            run_loss += loss.item() * inputs.size(0)
            _, pred   = torch.max(out, 1)
            total    += labels.size(0)
            correct  += (pred == labels).sum().item()

        t_loss = run_loss / total
        t_acc  = correct  / total

        # ── Validate ──────────────────────────────────────────────────────────
        model.eval()
        v_loss = v_correct = v_total = 0
        with torch.no_grad():
            for inputs, labels, _ in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                out   = model(inputs)
                loss  = criterion(out, labels)
                v_loss    += loss.item() * inputs.size(0)
                _, pred    = torch.max(out, 1)
                v_total   += labels.size(0)
                v_correct += (pred == labels).sum().item()

        v_loss_avg = v_loss    / v_total
        v_acc      = v_correct / v_total

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss_avg)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)

        print(f"\r        Epoch {epoch+1:02d}/{NUM_EPOCHS}  "
              f"Train Loss={t_loss:.4f} Acc={t_acc:.4f}  "
              f"Val Loss={v_loss_avg:.4f} Acc={v_acc:.4f}   ",
              end="")

    print()    # newline after epoch loop

    # ── Save model weights ────────────────────────────────────────────────────
    model_path = os.path.join(save_dir, "model.pth")
    torch.save(model.state_dict(), model_path)
    print(f"        ✔ Model saved → {model_path}")

    # ── Save model config ─────────────────────────────────────────────────────
    config = {
        "model_name":  model_name,
        "task":        task_name,
        "phoneme":     phoneme,
        "fold":        int(fold),
        "num_epochs":  NUM_EPOCHS,
        "batch_size":  BATCH_SIZE,
        "learning_rate": LR,
        "optimizer":   "Adam",
        "loss":        "CrossEntropyLoss",
        "input_size":  "3 × 224 × 224",
        "num_classes": 2,
        "class_map":   {"0": "Control", "1": "Dysarthric"},
        "device":      str(device),
        "final_train_loss": round(history['train_loss'][-1], 6),
        "final_val_loss":   round(history['val_loss'][-1],   6),
        "final_train_acc":  round(history['train_acc'][-1],  6),
        "final_val_acc":    round(history['val_acc'][-1],    6),
        "how_to_load": (
            "model = get_<arch>()  # build same architecture\n"
            "model.load_state_dict(torch.load('model.pth', map_location='cpu'))\n"
            "model.eval()"
        )
    }
    with open(os.path.join(save_dir, "model_config.txt"), "w") as f:
        f.write(json.dumps(config, indent=4))

    # ── Save training curves ──────────────────────────────────────────────────
    ep = range(1, NUM_EPOCHS + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(ep, history['train_loss'], label='Train', color='royalblue')
    axes[0].plot(ep, history['val_loss'],   label='Val',   color='coral')
    axes[0].set_title('Loss over Epochs')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(ep, history['train_acc'], label='Train', color='royalblue')
    axes[1].plot(ep, history['val_acc'],   label='Val',   color='coral')
    axes[1].set_title('Accuracy over Epochs')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle(f'{model_name} | {task_name} | {phoneme} | Fold {fold}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "training_curves.png"), dpi=150, bbox_inches='tight')
    plt.close()

    # ── Collect final val predictions ─────────────────────────────────────────
    val_probs, val_labels_dict = {}, {}
    model.eval()
    with torch.no_grad():
        for inputs, labels, names in val_loader:
            inputs = inputs.to(device)
            out    = model(inputs)
            probs  = torch.softmax(out, dim=1).cpu().numpy()
            for i, name in enumerate(names):
                val_probs[name]       = probs[i]
                val_labels_dict[name] = int(labels[i].item())

    return val_probs, val_labels_dict

# ==============================================================================
# SECTION 6 — SAVE PROBABILITY VECTORS
# ==============================================================================

def save_probability_csv(val_probs:  dict,
                         val_labels: dict,
                         save_dir:   str,
                         filename:   str = "val_probabilities.csv") -> None:
    """
    Saves a CSV with one row per validation sample:

    Columns
    ───────
      sample_name     : base filename (no extension)
      true_label      : 0 = Control, 1 = Dysarthric
      true_class      : human-readable label
      prob_control    : softmax probability for class 0
      prob_dysarthric : softmax probability for class 1
      predicted_label : argmax of probabilities
      predicted_class : human-readable prediction
      correct         : 1 if prediction matches true label, else 0
    """
    os.makedirs(save_dir, exist_ok=True)
    rows = []
    label_map = {0: "Control", 1: "Dysarthric"}

    for name in sorted(val_probs.keys()):
        probs    = val_probs[name]
        true_lbl = val_labels[name]
        pred_lbl = int(np.argmax(probs))
        rows.append({
            "sample_name":     name,
            "true_label":      true_lbl,
            "true_class":      label_map.get(true_lbl, str(true_lbl)),
            "prob_control":    round(float(probs[0]), 6),
            "prob_dysarthric": round(float(probs[1]), 6),
            "predicted_label": pred_lbl,
            "predicted_class": label_map.get(pred_lbl, str(pred_lbl)),
            "correct":         int(pred_lbl == true_lbl),
        })

    df = pd.DataFrame(rows)
    out_path = os.path.join(save_dir, filename)
    df.to_csv(out_path, index=False)
    print(f"        ✔ Probabilities saved → {out_path}  ({len(df)} samples)")

# ==============================================================================
# SECTION 7 — METRICS
# ==============================================================================

def compute_metrics(y_true, y_probs) -> dict:
    """
    y_true  : list[int]
    y_probs : list[np.ndarray shape (2,)]
    Returns dict of all performance metrics + plot-ready arrays.
    """
    y_true  = np.array(y_true,  dtype=int)
    y_probs = np.array(y_probs, dtype=float)
    y_pred  = np.argmax(y_probs, axis=1)
    y_pos   = y_probs[:, 1]

    acc = accuracy_score(y_true, y_pred)
    n   = len(y_true)
    ci  = 1.96 * math.sqrt(acc * (1 - acc) / n) if n > 1 else 0.0

    cm             = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sensitivity    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity    = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    fpr, tpr, _    = roc_curve(y_true, y_pos, pos_label=1)
    roc_auc        = auc(fpr, tpr)

    prec, rec, _   = precision_recall_curve(y_true, y_pos, pos_label=1)
    pr_auc         = average_precision_score(y_true, y_pos)

    f1             = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    report         = classification_report(
                         y_true, y_pred,
                         target_names=["Control", "Dysarthric"],
                         zero_division=0)

    return dict(
        accuracy=acc, ci_95=ci,
        roc_auc=roc_auc, pr_auc=pr_auc,
        sensitivity=sensitivity, specificity=specificity,
        f1=f1, cm=cm,
        fpr=fpr, tpr=tpr,
        precision=prec, recall=rec,
        y_true=y_true, y_pred=y_pred,
        report=report
    )

# ==============================================================================
# SECTION 8 — SAVE EVALUATION PLOTS & TEXT
# ==============================================================================

def save_evaluation(metrics: dict, save_dir: str, title: str) -> None:
    """
    Saves to save_dir:
      metrics.txt | confusion_matrix.png | roc_curve.png | pr_curve.png
    """
    os.makedirs(save_dir, exist_ok=True)
    m = metrics

    # Metrics text
    with open(os.path.join(save_dir, "metrics.txt"), "w") as f:
        f.write(f"=== {title} ===\n\n")
        f.write(f"Accuracy    : {m['accuracy']:.4f}  (95% CI ±{m['ci_95']:.4f})\n")
        f.write(f"ROC AUC     : {m['roc_auc']:.4f}\n")
        f.write(f"PR AUC      : {m['pr_auc']:.4f}\n")
        f.write(f"Sensitivity : {m['sensitivity']:.4f}\n")
        f.write(f"Specificity : {m['specificity']:.4f}\n")
        f.write(f"F1 Score    : {m['f1']:.4f}\n\n")
        f.write("Classification Report:\n")
        f.write(m['report'])

    # Confusion matrix
    plt.figure(figsize=(6, 5))
    sns.heatmap(m['cm'], annot=True, fmt='d', cmap='Blues',
                xticklabels=['Control', 'Dysarthric'],
                yticklabels=['Control', 'Dysarthric'])
    plt.title(f'{title}\nConfusion Matrix', fontsize=10)
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "confusion_matrix.png"), dpi=150, bbox_inches='tight')
    plt.close()

    # ROC curve
    plt.figure(figsize=(7, 6))
    plt.plot(m['fpr'], m['tpr'], color='darkorange', lw=2,
             label=f"AUC = {m['roc_auc']:.4f}")
    plt.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{title}\nROC Curve', fontsize=10)
    plt.legend(loc='lower right')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "roc_curve.png"), dpi=150, bbox_inches='tight')
    plt.close()

    # PR curve
    plt.figure(figsize=(7, 6))
    plt.plot(m['recall'], m['precision'], color='purple', lw=2,
             label=f"AUC = {m['pr_auc']:.4f}")
    plt.xlabel('Recall (Sensitivity)')
    plt.ylabel('Precision')
    plt.title(f'{title}\nPrecision–Recall Curve', fontsize=10)
    plt.legend(loc='lower left')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "pr_curve.png"), dpi=150, bbox_inches='tight')
    plt.close()

# ==============================================================================
# SECTION 9 — COMPARISON & AVERAGE TABLES
# ==============================================================================

_MODEL_COLORS = {
    "EfficientNet-B0": "#D6EAF8",   # light blue
    "ConvNeXt-Tiny":   "#FDEBD0",   # light orange
}
_HEADER_COLOR = "#2E4057"

METRIC_COLS   = ['accuracy', 'roc_auc', 'sensitivity', 'specificity', 'f1', 'pr_auc']
METRIC_LABELS = ["Accuracy", "ROC AUC", "Sensitivity\n(TPR)", "Specificity\n(TNR)",
                 "F1 Score\n(Weighted)", "PR AUC"]


def _make_table_image(records: list, save_path: str, title: str) -> None:
    """Render a styled matplotlib table and save as PNG."""
    if not records:
        return

    df = pd.DataFrame(records)
    col_labels = ["Phoneme / Token", "Model"] + METRIC_LABELS

    cell_text = []
    for _, row in df.iterrows():
        cell_text.append([
            str(row.get('phoneme',     '')),
            str(row.get('model',       '')),
            f"{row.get('accuracy',    0):.4f}",
            f"{row.get('roc_auc',     0):.4f}",
            f"{row.get('sensitivity', 0):.4f}",
            f"{row.get('specificity', 0):.4f}",
            f"{row.get('f1',          0):.4f}",
            f"{row.get('pr_auc',      0):.4f}",
        ])

    n_rows = len(cell_text)
    fig_h  = max(3.0, n_rows * 0.42 + 2.0)
    fig, ax = plt.subplots(figsize=(17, fig_h))
    ax.axis('off')

    tbl = ax.table(cellText=cell_text, colLabels=col_labels,
                   cellLoc='center', loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.5)

    col_widths = [0.14, 0.14, 0.10, 0.10, 0.12, 0.12, 0.14, 0.10]
    for j, w in enumerate(col_widths):
        for i in range(n_rows + 1):
            tbl[(i, j)].set_width(w)

    # Header styling
    for j in range(len(col_labels)):
        cell = tbl[(0, j)]
        cell.set_facecolor(_HEADER_COLOR)
        cell.set_text_props(color='white', fontweight='bold', fontsize=8.5)
        cell.set_height(0.08)

    # Per-phoneme best-value highlighting
    best_map = {}
    for phoneme_val in df['phoneme'].unique():
        sub = df[df['phoneme'] == phoneme_val]
        for mc in METRIC_COLS:
            best_map[(phoneme_val, mc)] = sub[mc].max()

    metric_col_idx = {mc: i + 2 for i, mc in enumerate(METRIC_COLS)}

    for i, (_, row) in enumerate(df.iterrows()):
        model_name = row.get('model', '')
        bg = _MODEL_COLORS.get(model_name, '#FFFFFF')
        for j in range(len(col_labels)):
            c = tbl[(i + 1, j)]
            c.set_facecolor(bg)
            c.set_edgecolor('#CCCCCC')

        for mc, col_j in metric_col_idx.items():
            best_val = best_map.get((row.get('phoneme'), mc), None)
            if best_val is not None and abs(row.get(mc, -1) - best_val) < 1e-6:
                tbl[(i + 1, col_j)].set_text_props(fontweight='bold')
                tbl[(i + 1, col_j)].set_edgecolor('#E74C3C')
                tbl[(i + 1, col_j)].set_linewidth(1.5)

    patches = [
        mpatches.Patch(color=_MODEL_COLORS["EfficientNet-B0"], label="EfficientNet-B0"),
        mpatches.Patch(color=_MODEL_COLORS["ConvNeXt-Tiny"],   label="ConvNeXt-Tiny"),
        mpatches.Patch(edgecolor='#E74C3C', facecolor='white', linewidth=1.5,
                       label="Best per phoneme"),
    ]
    plt.legend(handles=patches, loc='upper right', fontsize=7.5,
               bbox_to_anchor=(1.0, 1.0))

    plt.title(title, fontsize=12, fontweight='bold', pad=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"      ✔ Table saved → {save_path}")


def save_comparison_table(records: list, save_dir: str, title: str) -> None:
    """Save CSV + styled PNG comparison table."""
    if not records:
        return
    os.makedirs(save_dir, exist_ok=True)
    df       = pd.DataFrame(records)
    csv_name = title.replace(" ", "_").replace("—", "-").replace("/", "-")
    df.to_csv(os.path.join(save_dir, f"{csv_name}.csv"), index=False)
    _make_table_image(records,
                      os.path.join(save_dir, f"{csv_name}_table.png"),
                      title)


def save_average_metrics(records: list, save_dir: str, task_name: str) -> None:
    """
    Compute and save average metrics across all phonemes within a task,
    grouped by model.  Produces:
      average_metrics.csv
      average_metrics_table.png
      average_metrics_bar_chart.png
    """
    if not records:
        return
    os.makedirs(save_dir, exist_ok=True)

    df     = pd.DataFrame(records)
    avg_df = df.groupby('model')[METRIC_COLS].mean().reset_index()
    avg_df.insert(0, 'task', task_name)
    avg_df.to_csv(os.path.join(save_dir, "average_metrics.csv"), index=False)

    # Styled table
    avg_records = [{
        'phoneme': f"AVG ({task_name})",
        'model':   row['model'],
        **{mc: row[mc] for mc in METRIC_COLS}
    } for _, row in avg_df.iterrows()]
    _make_table_image(
        avg_records,
        os.path.join(save_dir, "average_metrics_table.png"),
        f"{task_name.capitalize()} — Average Metrics Across All Phonemes"
    )

    # Bar chart
    x      = np.arange(len(METRIC_COLS))
    width  = 0.35
    colors = list(_MODEL_COLORS.values())

    fig, ax = plt.subplots(figsize=(13, 6))
    for i, (_, row) in enumerate(avg_df.iterrows()):
        vals = [row[mc] for mc in METRIC_COLS]
        bars = ax.bar(x + i * width, vals, width,
                      label=row['model'], color=colors[i % len(colors)],
                      edgecolor='grey', linewidth=0.5)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.005,
                    f"{val:.3f}", ha='center', va='bottom', fontsize=7.5)

    ax.set_xticks(x + width * (len(avg_df) - 1) / 2)
    ax.set_xticklabels(METRIC_LABELS, fontsize=9)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel('Score', fontsize=10)
    ax.set_title(f"{task_name.capitalize()} — Average Metrics per Model\n"
                 f"(mean across all phonemes/tokens)",
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "average_metrics_bar_chart.png"),
                dpi=180, bbox_inches='tight')
    plt.close()
    print(f"      ✔ Average metrics saved → {save_dir}")

# ==============================================================================
# SECTION 10 — TASK DEFINITIONS
# ==============================================================================

TASKS = [
    {
        "task_name": "vowels",
        "csv_2d":    VOWELS_CSV_2D,
        "phonemes":  ["A", "E", "I", "O", "U"],
    },
    {
        "task_name": "words",
        "csv_2d":    WORDS_CSV_2D,
        "phonemes":  ["PAPA", "PIPI", "PUPU"],
    },
    {
        "task_name": "sentences",
        "csv_2d":    SENTENCES_CSV_2D,
        "phonemes":  ["S0", "S1"],
    },
]

# ==============================================================================
# SECTION 11 — MAIN LOOP
# ==============================================================================

os.makedirs(RESULTS_BASE, exist_ok=True)

grand_records = []      # accumulates all tasks × phonemes × models

for task in TASKS:
    task_name = task["task_name"]

    print(f"\n{'='*70}")
    print(f"  TASK : {task_name.upper()}")
    print(f"{'='*70}")

    # Load and preprocess CSV
    df_2d = pd.read_csv(task["csv_2d"])
    df_2d["Phoneme"] = df_2d["Phoneme"].astype(str).str.upper().str.strip()
    df_2d["Label"]   = df_2d["Label"].apply(
        lambda x: 1 if str(x).strip().lower() == "dysarthric" else 0
    )

    task_records = []

    # ── Phoneme loop ──────────────────────────────────────────────────────────
    for phoneme in task["phonemes"]:

        df_p = df_2d[df_2d["Phoneme"] == phoneme]
        if df_p.empty:
            print(f"\n  ⚠  No data for phoneme {phoneme} — skipping.")
            continue

        print(f"\n  {'─'*60}")
        print(f"  Phoneme/Token : {phoneme}  (rows = {len(df_p)})")
        print(f"  {'─'*60}")

        token_dir = os.path.join(RESULTS_BASE, task_name, f"Token_{phoneme}")

        # ── Model loop ────────────────────────────────────────────────────────
        for model_name, model_factory in MODEL_REGISTRY.items():

            print(f"\n    ── Model : {model_name} ──")

            model_dir = os.path.join(token_dir, model_name.replace("-", "_"))
            os.makedirs(model_dir, exist_ok=True)

            # Fold-level accumulators
            all_true, all_probs     = [], []
            all_prob_rows           = []     # for overall_probabilities.csv

            # ── Fold loop ─────────────────────────────────────────────────────
            for fold in sorted(df_p["Fold"].unique()):
                print(f"\n      Fold {fold}")

                fold_df = df_p[df_p["Fold"] == fold]
                tr_df   = fold_df[fold_df["Type"].str.lower() == "train"]
                va_df   = fold_df[fold_df["Type"].str.lower() == "validation"]

                if tr_df.empty or va_df.empty:
                    print(f"        ⚠  Fold {fold}: a split is empty — skipping.")
                    continue

                _pin = torch.cuda.is_available()
                tl = DataLoader(SpectrogramDataset(tr_df),
                                batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=2, pin_memory=_pin)
                vl = DataLoader(SpectrogramDataset(va_df),
                                batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=2, pin_memory=_pin)

                fold_dir = os.path.join(model_dir, f"Fold_{fold}")
                os.makedirs(fold_dir, exist_ok=True)

                # Train — saves model.pth + model_config.txt + training_curves.png
                model_instance = model_factory()
                val_probs, val_labels = train_model(
                    model_instance, tl, vl,
                    model_name, fold_dir,
                    task_name, phoneme, fold
                )
                del model_instance
                torch.cuda.empty_cache()

                common = sorted(set(val_probs) & set(val_labels))
                if not common:
                    print("        ⚠  No common samples — skipping fold evaluation.")
                    continue

                f_true  = [val_labels[n] for n in common]
                f_probs = [val_probs[n]  for n in common]

                # Save per-fold probability CSV
                save_probability_csv(
                    {n: val_probs[n]  for n in common},
                    {n: val_labels[n] for n in common},
                    fold_dir,
                    filename="val_probabilities.csv"
                )

                # Evaluate and save fold plots + metrics
                fold_metrics = compute_metrics(f_true, f_probs)
                save_evaluation(
                    fold_metrics, fold_dir,
                    f"{phoneme} Fold {fold} — {model_name}"
                )

                # Accumulate for OVERALL
                all_true.extend(f_true)
                all_probs.extend(f_probs)

                # Accumulate probability rows with fold column for overall CSV
                label_map = {0: "Control", 1: "Dysarthric"}
                for n in common:
                    probs_arr = val_probs[n]
                    pred_lbl  = int(np.argmax(probs_arr))
                    all_prob_rows.append({
                        "fold":            int(fold),
                        "sample_name":     n,
                        "true_label":      val_labels[n],
                        "true_class":      label_map.get(val_labels[n], str(val_labels[n])),
                        "prob_control":    round(float(probs_arr[0]), 6),
                        "prob_dysarthric": round(float(probs_arr[1]), 6),
                        "predicted_label": pred_lbl,
                        "predicted_class": label_map.get(pred_lbl, str(pred_lbl)),
                        "correct":         int(pred_lbl == val_labels[n]),
                    })

            # END fold loop ───────────────────────────────────────────────────

            if not all_true:
                print(f"\n    ⚠  No predictions collected for {phoneme} / {model_name}.")
                continue

            # ── OVERALL evaluation (all folds concatenated) ───────────────────
            print(f"\n      Computing OVERALL metrics …")
            overall_dir = os.path.join(model_dir, "OVERALL")
            os.makedirs(overall_dir, exist_ok=True)

            # Save overall probability CSV (all folds, with fold column)
            overall_prob_path = os.path.join(overall_dir, "overall_probabilities.csv")
            pd.DataFrame(all_prob_rows).to_csv(overall_prob_path, index=False)
            print(f"        ✔ Overall probabilities saved → {overall_prob_path}")

            overall_m = compute_metrics(all_true, all_probs)
            save_evaluation(
                overall_m, overall_dir,
                f"{phoneme} OVERALL — {model_name}"
            )

            rec = {
                "task":        task_name,
                "phoneme":     phoneme,
                "model":       model_name,
                "accuracy":    overall_m["accuracy"],
                "roc_auc":     overall_m["roc_auc"],
                "sensitivity": overall_m["sensitivity"],
                "specificity": overall_m["specificity"],
                "f1":          overall_m["f1"],
                "pr_auc":      overall_m["pr_auc"],
            }
            task_records.append(rec)
            grand_records.append(rec)

        # ── Per-phoneme comparison (both models side-by-side) ─────────────────
        phoneme_recs = [r for r in task_records if r["phoneme"] == phoneme]
        save_comparison_table(
            phoneme_recs,
            os.path.join(token_dir, "comparison"),
            f"{task_name.capitalize()} — {phoneme} — Model Comparison"
        )

    # END phoneme loop ─────────────────────────────────────────────────────────

    if task_records:
        # Per-task table (all phonemes × both models)
        save_comparison_table(
            task_records,
            os.path.join(RESULTS_BASE, task_name, "TASK_COMPARISON"),
            f"{task_name.capitalize()} — All Tokens — Model Comparison"
        )
        # Per-task average metrics
        save_average_metrics(
            task_records,
            os.path.join(RESULTS_BASE, task_name, "TASK_AVERAGE_METRICS"),
            task_name
        )

# END task loop ────────────────────────────────────────────────────────────────

# ==============================================================================
# SECTION 12 — GRAND OUTPUTS (all tasks + both models)
# ==============================================================================

if grand_records:

    # Grand comparison table
    display_records = []
    for r in grand_records:
        dr          = r.copy()
        dr["phoneme"] = f"[{r['task'].upper()}] {r['phoneme']}"
        display_records.append(dr)

    save_comparison_table(
        display_records,
        os.path.join(RESULTS_BASE, "GRAND_COMPARISON"),
        "Grand Comparison — All Tasks & Models"
    )

    # Grand average per task × model
    grand_avg_dir = os.path.join(RESULTS_BASE, "GRAND_AVERAGE_METRICS")
    os.makedirs(grand_avg_dir, exist_ok=True)

    grand_df  = pd.DataFrame(grand_records)
    grand_avg = grand_df.groupby(['task', 'model'])[METRIC_COLS].mean().reset_index()
    grand_avg.to_csv(os.path.join(grand_avg_dir, "grand_average_metrics.csv"), index=False)

    tasks_list  = grand_avg['task'].unique()
    models_list = list(MODEL_REGISTRY.keys())
    colors      = list(_MODEL_COLORS.values())

    for mc, ml in zip(METRIC_COLS, METRIC_LABELS):
        fig, ax = plt.subplots(figsize=(10, 5))
        x = np.arange(len(tasks_list))
        w = 0.35

        for mi, model_name in enumerate(models_list):
            vals = []
            for t in tasks_list:
                row = grand_avg[
                    (grand_avg['task'] == t) & (grand_avg['model'] == model_name)
                ]
                vals.append(float(row[mc].values[0]) if not row.empty else 0.0)

            bars = ax.bar(x + mi * w, vals, w,
                          label=model_name, color=colors[mi],
                          edgecolor='grey', linewidth=0.5)
            for bar, val in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.005,
                        f"{val:.3f}", ha='center', va='bottom', fontsize=8)

        ax.set_xticks(x + w * (len(models_list) - 1) / 2)
        ax.set_xticklabels([t.capitalize() for t in tasks_list], fontsize=10)
        ax.set_ylim(0, 1.12)
        ax.set_ylabel('Score', fontsize=10)
        ax.set_title(f"Grand Average — {ml}\n(mean per task × model)",
                     fontsize=12, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(grand_avg_dir, f"grand_avg_{mc}.png"),
                    dpi=180, bbox_inches='tight')
        plt.close()

    print(f"  ✔ Grand average charts saved → {grand_avg_dir}")

print(f"\n{'='*70}")
print("    ALL DONE")
print(f"  Results saved under: {RESULTS_BASE}")
print(f"{'='*70}")

# ==============================================================================
# HOW TO RELOAD A SAVED MODEL
# ==============================================================================
#
#  from torchvision import models
#  import torch, torch.nn as nn
#
#  def get_efficientnet_b0():
#      m = models.efficientnet_b0(weights=None)
#      m.classifier = nn.Sequential(nn.Dropout(0.3, inplace=True), nn.Linear(1280, 2))
#      return m
#
#  def get_convnext_tiny():
#      m = models.convnext_tiny(weights=None)
#      m.classifier[2] = nn.Linear(768, 2)
#      return m
#
#  model = get_efficientnet_b0()   # or get_convnext_tiny()
#  model.load_state_dict(torch.load("path/to/model.pth", map_location="cpu"))
#  model.eval()
#
# ==============================================================================

Using device: cuda

  TASK : VOWELS

  ────────────────────────────────────────────────────────────
  Phoneme/Token : A  (rows = 2000)
  ────────────────────────────────────────────────────────────

    ── Model : EfficientNet-B0 ──

      Fold 1
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 193MB/s]


      [EfficientNet-B0] training …
        Epoch 30/30  Train Loss=0.1615 Acc=0.9706  Val Loss=0.1446 Acc=0.9643   
        ✔ Model saved → /content/drive/MyDrive/ICMR/Results_EfficientNet_ConvNeXt/vowels/Token_A/EfficientNet_B0/Fold_1/model.pth
        ✔ Probabilities saved → /content/drive/MyDrive/ICMR/Results_EfficientNet_ConvNeXt/vowels/Token_A/EfficientNet_B0/Fold_1/val_probabilities.csv  (14 samples)

      Fold 2
      [EfficientNet-B0] training …
        Epoch 30/30  Train Loss=0.1391 Acc=0.9790  Val Loss=0.3716 Acc=0.7812   
        ✔ Model saved → /content/drive/MyDrive/ICMR/Results_EfficientNet_ConvNeXt/vowels/Token_A/EfficientNet_B0/Fold_2/model.pth
        ✔ Probabilities saved → /content/drive/MyDrive/ICMR/Results_EfficientNet_ConvNeXt/vowels/Token_A/EfficientNet_B0/Fold_2/val_probabilities.csv  (16 samples)

      Fold 3
      [EfficientNet-B0] training …
        Epoch 30/30  Train Loss=0.1236 Acc=0.9821  Val Loss=0.2824 Acc=0.8438   
        ✔ Model saved → /content/dri

100%|██████████| 109M/109M [00:00<00:00, 138MB/s]


      [ConvNeXt-Tiny] training …
        Epoch 30/30  Train Loss=0.0126 Acc=0.9941  Val Loss=0.3081 Acc=0.8571   
        ✔ Model saved → /content/drive/MyDrive/ICMR/Results_EfficientNet_ConvNeXt/vowels/Token_A/ConvNeXt_Tiny/Fold_1/model.pth
        ✔ Probabilities saved → /content/drive/MyDrive/ICMR/Results_EfficientNet_ConvNeXt/vowels/Token_A/ConvNeXt_Tiny/Fold_1/val_probabilities.csv  (14 samples)

      Fold 2
      [ConvNeXt-Tiny] training …
        Epoch 30/30  Train Loss=0.0051 Acc=1.0000  Val Loss=0.8998 Acc=0.7812   
        ✔ Model saved → /content/drive/MyDrive/ICMR/Results_EfficientNet_ConvNeXt/vowels/Token_A/ConvNeXt_Tiny/Fold_2/model.pth
        ✔ Probabilities saved → /content/drive/MyDrive/ICMR/Results_EfficientNet_ConvNeXt/vowels/Token_A/ConvNeXt_Tiny/Fold_2/val_probabilities.csv  (16 samples)

      Fold 3
      [ConvNeXt-Tiny] training …
        Epoch 30/30  Train Loss=0.0052 Acc=1.0000  Val Loss=0.6318 Acc=0.8125   
        ✔ Model saved → /content/drive/MyDrive/ICM